In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Geometry-V7 R3 resumable method iteration
Inspect complete artifacts first, preserve every partial path, and invoke the exact-bound runner at most once.

In [ ]:
import json
import re
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
EXECUTION_EXACT = '896571fd17fbc161bbb617f74677328a012ce43a'
NOTEBOOK_DELIVERY_EXACT = '3b69add60e851fbda3af06f80b4f626e77e992d6'
RESULT_SCHEMA = 'geometry_v7_r3_exploratory_result_v1'
COMPLETE_STATUSES = {'R3_METHOD_IMPROVED', 'R3_METHOD_NOT_IMPROVED', 'OPERATIONAL_FAILURE'}

def inspect_result(root, execution_exact):
    root = Path(root)
    if not root.exists():
        return {'state': 'ABSENT', 'payload': None, 'reason': 'path_absent', 'path': str(root)}
    if not root.is_dir():
        return {'state': 'PARTIAL', 'payload': None, 'reason': 'path_exists_not_directory', 'path': str(root)}
    result = root / 'result.json'
    if not result.is_file():
        return {'state': 'PARTIAL', 'payload': None, 'reason': 'result_json_absent', 'path': str(root)}
    try:
        payload = json.loads(result.read_text(encoding='utf-8'))
    except Exception as error:
        return {'state': 'PARTIAL', 'payload': None, 'reason': f'result_json_unreadable:{type(error).__name__}', 'path': str(root)}
    if not isinstance(payload, dict):
        return {'state': 'PARTIAL', 'payload': None, 'reason': 'result_json_not_object', 'path': str(root)}
    if payload.get('schema') != RESULT_SCHEMA:
        return {'state': 'PARTIAL', 'payload': payload, 'reason': 'schema_mismatch', 'path': str(root)}
    if payload.get('exact') != execution_exact:
        return {'state': 'PARTIAL', 'payload': payload, 'reason': 'exact_mismatch', 'path': str(root)}
    if payload.get('status') not in COMPLETE_STATUSES:
        return {'state': 'PARTIAL', 'payload': payload, 'reason': 'status_invalid', 'path': str(root)}
    return {'state': 'COMPLETE', 'payload': payload, 'reason': None, 'path': str(root)}

def fresh_unique_path(base):
    base = Path(base)
    for index in range(1, 10000):
        candidate = base.with_name(f'{base.name}.resume-{index:03d}')
        if not candidate.exists():
            return candidate
    raise RuntimeError(f'no fresh resume path available beside {base}')

def local_result_candidates(local_base):
    local_base = Path(local_base)
    resumes = sorted(local_base.parent.glob(f'{local_base.name}.resume-*'), key=lambda path: path.name)
    return (local_base, *resumes)

def plan_result_state(drive_root, local_base, checkpoint_base, execution_exact):
    drive_state = inspect_result(drive_root, execution_exact)
    candidate_states = tuple(inspect_result(path, execution_exact) for path in local_result_candidates(local_base))
    complete_local = next((state for state in candidate_states if state['state'] == 'COMPLETE'), None)
    local_state = complete_local or candidate_states[0]
    plan = {'DRIVE_STATE': drive_state, 'LOCAL_STATE': local_state,
            'LOCAL_RESULT_DIR': Path(local_state['path']), 'SYNCSEAL_CHECKPOINT': Path(checkpoint_base),
            'LOCAL_CANDIDATE_STATES': candidate_states,
            'RUN_REQUIRED': False, 'PUBLISH_REQUIRED': False}
    if drive_state['state'] == 'PARTIAL':
        raise RuntimeError(f"Drive result conflict at {drive_state['path']}: {drive_state['reason']}")
    if drive_state['state'] == 'COMPLETE':
        return plan
    if local_state['state'] == 'COMPLETE':
        plan['PUBLISH_REQUIRED'] = True
        return plan
    residual = any(state['state'] == 'PARTIAL' for state in candidate_states) or Path(checkpoint_base).exists()
    if residual:
        plan['LOCAL_RESULT_DIR'] = fresh_unique_path(local_base)
        plan['SYNCSEAL_CHECKPOINT'] = fresh_unique_path(checkpoint_base)
    plan['RUN_REQUIRED'] = True
    plan['PUBLISH_REQUIRED'] = True
    return plan

def inspect_checkout(path, execution_exact, run=subprocess.run):
    path = Path(path)
    if not path.exists():
        return {'state': 'ABSENT', 'reason': 'path_absent', 'path': str(path)}
    if not path.is_dir():
        return {'state': 'INVALID', 'reason': 'path_exists_not_directory', 'path': str(path)}
    try:
        exact = run(['git', '-C', str(path), 'rev-parse', 'HEAD'], check=True, text=True, capture_output=True).stdout.strip()
        dirty = run(['git', '-C', str(path), 'status', '--porcelain'], check=True, text=True, capture_output=True).stdout
    except Exception as error:
        return {'state': 'INVALID', 'reason': f'git_inspection_failed:{type(error).__name__}', 'path': str(path)}
    if exact == execution_exact and not dirty:
        return {'state': 'REUSABLE', 'reason': None, 'path': str(path)}
    return {'state': 'INVALID', 'reason': 'exact_or_cleanliness_mismatch', 'path': str(path)}

def plan_checkout(base, execution_exact, run=subprocess.run):
    state = inspect_checkout(base, execution_exact, run=run)
    if state['state'] == 'REUSABLE':
        return {'path': Path(base), 'reuse': True, 'prior': state}
    if state['state'] == 'ABSENT':
        return {'path': Path(base), 'reuse': False, 'prior': state}
    isolated_base = Path(base).with_name(f'{Path(base).name}.isolated')
    return {'path': fresh_unique_path(isolated_base), 'reuse': False, 'prior': state}

def validate_runner_completion(returncode, local_result, execution_exact):
    if returncode not in (0, 2):
        raise RuntimeError(f'Geometry-V7 R3 runner returned unexpected code {returncode}')
    state = inspect_result(local_result, execution_exact)
    if state['state'] != 'COMPLETE':
        raise RuntimeError(f"R3 runner result incomplete at {state['path']}: {state['reason']}")
    return state

if re.fullmatch(r'[0-9a-f]{40}', EXECUTION_EXACT) is None:
    raise RuntimeError('Geometry-V7 R3 execution exact is malformed')
R1A_ARTIFACT_ROOT = Path('/content/drive/MyDrive/CEG-WM/Geometry-V7/ac590330e91aacf4b3283df1e94572a0e4f983a0/r1a-f2')
R1B_REPAIR_ARTIFACT_ROOT = Path('/content/drive/MyDrive/CEG-WM/Geometry-V7/3b9819d80b07704a4caab8b7aaa581cf9eb8a3c5/r1b-repair')
R2_ARTIFACT_ROOT = Path('/content/drive/MyDrive/CEG-WM/Geometry-V7/ffac9d4c1e575c27240d9423bbd30e0713aa2dcd/r2-selective')
DRIVE_RESULT_DIR = Path('/content/drive/MyDrive/CEG-WM/Geometry-V7') / EXECUTION_EXACT / 'r3-exploratory'
LOCAL_RESULT_BASE = Path('/content/geometry_v7_r3_exploratory_result')
CHECKPOINT_BASE = Path('/content/checkpoints/r3_syncmodel.jit.pt')
CHECKOUT_BASE = Path('/content/CEG-WM')
STATE_PLAN = plan_result_state(DRIVE_RESULT_DIR, LOCAL_RESULT_BASE, CHECKPOINT_BASE, EXECUTION_EXACT)
DRIVE_STATE = STATE_PLAN['DRIVE_STATE']
LOCAL_STATE = STATE_PLAN['LOCAL_STATE']
LOCAL_RESULT_DIR = STATE_PLAN['LOCAL_RESULT_DIR']
SYNCSEAL_CHECKPOINT = STATE_PLAN['SYNCSEAL_CHECKPOINT']
RUN_REQUIRED = STATE_PLAN['RUN_REQUIRED']
PUBLISH_REQUIRED = STATE_PLAN['PUBLISH_REQUIRED']
checkout = CHECKOUT_BASE
CHECKOUT_REUSED = False
if DRIVE_STATE['state'] == 'COMPLETE':
    print('existing_drive_artifacts_ready', DRIVE_STATE['payload']['status'], DRIVE_RESULT_DIR)
elif LOCAL_STATE['state'] == 'COMPLETE':
    print('existing_local_artifacts_ready', LOCAL_STATE['payload']['status'], LOCAL_RESULT_DIR)
else:
    print('r3_run_required', LOCAL_RESULT_DIR, SYNCSEAL_CHECKPOINT)

In [ ]:
if not RUN_REQUIRED:
    print('runner_skipped_existing_complete_artifact')
else:
    import torch
    if not all(path.is_dir() for path in (R1A_ARTIFACT_ROOT, R1B_REPAIR_ARTIFACT_ROOT, R2_ARTIFACT_ROOT)):
        raise FileNotFoundError('fixed accepted R1A, R1B-repair, or R2 artifact is absent')
    assert torch.cuda.is_available(), 'GPU required for official SyncSeal D4 probes'
    CHECKOUT_PLAN = plan_checkout(CHECKOUT_BASE, EXECUTION_EXACT)
    checkout = CHECKOUT_PLAN['path']
    CHECKOUT_REUSED = CHECKOUT_PLAN['reuse']
    if not CHECKOUT_REUSED:
        subprocess.run(['git', 'clone', REPO_URL, str(checkout)], check=True)
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', EXECUTION_EXACT], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(checkout)], check=True)
    command = [
        sys.executable, '-m', 'experiments.run_geometry_v7_r3',
        '--repo-root', str(checkout), '--expected-exact', EXECUTION_EXACT,
        '--r1a-artifact-root', str(R1A_ARTIFACT_ROOT),
        '--r1b-repair-artifact-root', str(R1B_REPAIR_ARTIFACT_ROOT),
        '--r2-artifact-root', str(R2_ARTIFACT_ROOT),
        '--syncseal-checkpoint', str(SYNCSEAL_CHECKPOINT),
        '--result-dir', str(LOCAL_RESULT_DIR),
    ]
    completed = subprocess.run(command, cwd=checkout, text=True, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, check=False)
    print(completed.stdout.strip())
    LOCAL_STATE = validate_runner_completion(completed.returncode, LOCAL_RESULT_DIR, EXECUTION_EXACT)
    RUN_REQUIRED = False
    PUBLISH_REQUIRED = True
    print('new_local_artifacts_ready', LOCAL_STATE['payload']['status'], LOCAL_RESULT_DIR)

In [ ]:
import shutil

DRIVE_STATE = inspect_result(DRIVE_RESULT_DIR, EXECUTION_EXACT)
if DRIVE_STATE['state'] == 'COMPLETE':
    RUN_REQUIRED = False
    PUBLISH_REQUIRED = False
    print('existing_drive_artifacts_ready', DRIVE_STATE['payload']['status'], DRIVE_RESULT_DIR)
elif DRIVE_STATE['state'] == 'PARTIAL':
    raise RuntimeError(f"Drive result conflict at {DRIVE_STATE['path']}: {DRIVE_STATE['reason']}")
else:
    LOCAL_STATE = inspect_result(LOCAL_RESULT_DIR, EXECUTION_EXACT)
    if LOCAL_STATE['state'] != 'COMPLETE':
        raise RuntimeError(f"Local R3 result incomplete at {LOCAL_STATE['path']}: {LOCAL_STATE['reason']}")
    DRIVE_RESULT_DIR.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(LOCAL_RESULT_DIR, DRIVE_RESULT_DIR)
    DRIVE_STATE = inspect_result(DRIVE_RESULT_DIR, EXECUTION_EXACT)
    if DRIVE_STATE['state'] != 'COMPLETE':
        raise RuntimeError(f"Published R3 result incomplete at {DRIVE_STATE['path']}: {DRIVE_STATE['reason']}")
    RUN_REQUIRED = False
    PUBLISH_REQUIRED = False
    print('published_drive_artifacts_ready', DRIVE_STATE['payload']['status'], DRIVE_RESULT_DIR)